# HIT Spider
Init

In [1]:
import sys
import os
# Directory Management
try:
    # Run in Terminal
    ROOT_DIR = os.path.dirname(os.path.abspath(__file__))
except:
    # Run in ipykernel & interactive
    ROOT_DIR = os.getcwd()
if ".." not in sys.path:
    sys.path.append("..")

from fast_legged_planner_py.robot_interface.hitspider_robotinterface import HITSpider_RobotInterface, FOOT_LINK_NAME
from fast_legged_planner_py.robot_interface.pin_IK import pinIK
import pinocchio as pin
import numpy as np


urdf_file = os.path.join(os.path.dirname(ROOT_DIR), "model", "hit_spider", "urdf", "ElSpider.urdf")
pack_dirs = [os.path.dirname(os.path.dirname(ROOT_DIR))]
hit_spider = HITSpider_RobotInterface(urdf_file, pack_dirs)


/home/yip/miniconda3/envs/robotic_course/lib/python3.9/site-packages/scipy/__init__.py:132: UserWarning: A NumPy version >=1.21.6 and <1.28.0 is required for this version of SciPy (detected version 1.21.0)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


You can open the visualizer by visiting the following URL:
http://127.0.0.1:7000/static/


In [2]:
# Collision Model
# hit_spider.print_frames()
# hit_spider.print_joints()

# hit_spider.get_frameid(FOOT_LINK_NAME[2])

In [3]:
# Display
q0 = hit_spider.robot.q0
q_leg = [-2.56, -2.11, -0.484]
q = hit_spider.get_full_q(q_leg, 0)
hit_spider.viz.display(q)
# hit_spider.vis_collision_model(q0)
for link in FOOT_LINK_NAME:
    print(hit_spider.get_frame_placement(q0,link).translation)
# hit_spider.robot.computeFrameJacobian
# print(hit_spider.robot.data.oMi[3].translation)

[-0.54        0.93529727 -0.5       ]
[-0.54       -0.93529727 -0.5       ]
[-1.08000000e+00  5.01820807e-16 -5.00000000e-01]
[ 0.54        0.93529727 -0.5       ]
[ 0.54       -0.93529727 -0.5       ]
[ 1.08000000e+00 -1.99840144e-16 -5.00000000e-01]


In [17]:
## Inverse Kenematics (iterative)
# leg_num = 0
# q_leg = hit_spider.IK_foot(leg_num, np.array([0.1, 0.1, 0.1]))
# # print(q_leg)
# hit_spider.viz.display(hit_spider.get_full_q(q_leg,leg_num))

## Inverse Kinematics (IKFast single leg)
# index = 5
# q_leg = hit_spider.IKFast_foot(index, np.array([0.5,0, -0.5]))
# print(q_leg)
# q = hit_spider.get_full_q(q_leg, index)
# hit_spider.viz.display(q)

## Inverse Kinematics (IKFast all legs)
import time
use_ikfast = True
check_valid = True
fall_back = False
sample_num = 10000
amp = 3
h = -0.5
index = 0
sphere_size = 0.05
hit_spider.viz_clear()
if use_ikfast:
    IKs = hit_spider.IKFast_foots
    IK = hit_spider.IKFast_foot
else:
    IKs = lambda target_list, valid_check, fall_back: hit_spider.IK_foots(target_list)
    IK = lambda index, target, valid_check, fall_back: hit_spider.IK_foot(index, target)
def random_target():
    return (np.random.rand(3)-0.5)*amp

errcnt = 0
start = time.perf_counter()
for theta in np.linspace(0, 2*np.pi, sample_num):
    # target_list = [np.array([-amp*np.sin(theta), amp*np.cos(theta), h]) for i in range(6)]
    target_list = [random_target() for i in range(6)]
    try:
        if index is not None:
            q = IK(index, np.array(target_list[index]),
                   valid_check=check_valid, fall_back=fall_back)
            q = hit_spider.get_full_q(q, index)
            hit_spider.viz_add_sphere(target_list[index], sphere_size)
        else:
            q = IKs(target_list, valid_check=False, fall_back=False)
            for pos in target_list:
                hit_spider.viz_add_sphere(pos, sphere_size)
        hit_spider.viz.display(q)
    except Exception as e:
        # print(e)
        errcnt += 1
        pass
print("cost %s second" % (time.perf_counter() - start))
print("error count: %s" % errcnt)

cost 9.665304437279701 second
error count: 9026


array([-1.,  1., -1.])